### This script is purely for testing and comparison purposes against the async version. Not to be used in the pipeline.

In [0]:
import requests
import json
from typing import NamedTuple
from schemas import GeocodingResponse, GeocodingResult

In [0]:
GEOCODING_URL = "https://geocoding-api.open-meteo.com/v1/search"
CITY_DIM_TABLE = "workspace.open_meteo.dim_city"
CITY_SCHEMA = (
    "city_id BIGINT, name STRING, country STRING, country_code STRING, "
    "latitude DOUBLE, longitude DOUBLE, elevation DOUBLE"
)

class City(NamedTuple):
    name: str
    country_code: str


CITIES: list[City] = [
    City("London", "GB"),
    City("Paris", "FR"),
    City("Rome", "IT"),
    City("Madrid", "ES"),
    City("Berlin", "DE"),
    City("Moscow", "RU"),
    City("New York", "US"),
]

class GeocodingError(Exception):
    """Raised when a geocoding request or response validation fails."""

In [0]:
def geocode(city: City, *, timeout: float = 10.0) -> GeocodingResponse:
    """Fetch and validate geocoding metadata for a given city.

    Queries the geocoding API for the top matching location filtered by the
    city's ISO 3166-1 alpha-2 country code and validates the response payload.

    Parameters
    ----------
    city : City
        Named tuple containing the target city ``name`` and two-letter
        ``country`` code (e.g., ``City("London", "GB")``).
    timeout : float, default=10.0
        Maximum time in seconds to wait for the HTTP request before aborting.

    Returns
    -------
    GeocodingResponse
        Validated API response containing matching location records and
        query execution metadata.

    Raises
    ------
    GeocodingError
        If the HTTP request fails, returns a non-2xx status code, or produces
        a response body that fails JSON or schema validation.
    """
    try:
        response = requests.get(
            GEOCODING_URL,
            params={
                "name": city.name,
                "countryCode": city.country_code,
                "count": 1,
                "language": "en", 
                "format": "json",
            },
            timeout=timeout,
        )
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        raise GeocodingError(f"Error geocoding {city.name}: {e}") from e
    try:
        data = GeocodingResponse.model_validate_json(response.text)
    except json.decoder.JSONDecodeError as e:
        raise GeocodingError(f"Error parsing geocoding response: {e}") from e
    return data

In [0]:
locations = []
try:
    for city in CITIES:
        response = geocode(city).results[0]
        latitude = response.latitude
        longitude = response.longitude
        locations.append((latitude, longitude))
except GeocodingError as e:
    print(f"Error: {e}")
except Exception as e:
    print(f"Unexpected error: {e}")

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
# restart the Python process, then re-run your imports
dbutils.library.restartPython()

In [0]:
from open_meteo import geocode_cities, fetch_hourly_forecast, Location

res = await geocode_cities(CITIES)

In [0]:
locations = [Location(r.latitude, r.longitude) for r in res.values()]

In [0]:
res = await fetch_hourly_forecast(locations)

In [0]:
res